In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn import preprocessing
from dgl.data import DGLDataset
import dgl
import time
import networkx as nx
import category_encoders as ce
import torch.nn as nn
import torch.nn.functional as F
import dgl.function as fn
import torch
import tqdm
import math

from typing import *
from sklearn.preprocessing import StandardScaler, Normalizer
import socket
import struct
import random
from sklearn.model_selection import train_test_split
import gc

import torch.optim as optim
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score
from sklearn.ensemble import IsolationForest
import itertools
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

graphs, _ = dgl.load_graphs('train.bin')
train_g = graphs[0]
train_g = train_g.to(device)

class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out, activation):
      super(SAGELayer, self).__init__()
      self.W_apply = nn.Linear(ndim_in + ndim_out , ndim_out)
      self.activation = F.relu
      self.W_msg = nn.Linear(ndim_in + edims, ndim_out)
      self.W_edge = nn.Linear(ndim_out * 2 + edims, 64)
      self.reset_parameters()

    def reset_parameters(self):
      gain = nn.init.calculate_gain('relu')
      nn.init.xavier_uniform_(self.W_apply.weight, gain=gain)

    def message_func(self, edges):
      # return {'m':  edges.data['h']}
        return {'m': self.W_msg(torch.cat([edges.data['h'], edges.dst['h']], 2))}

    def forward(self, g_dgl, nfeats, efeats):
      with g_dgl.local_scope():
        g = g_dgl
        g.ndata['h'] = nfeats
        g.edata['h'] = efeats
        g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
        g.ndata['h'] = F.relu(self.W_apply(torch.cat([g.ndata['h'], g.ndata['h_neigh']], 2)))
        u, v = g.edges()
        src_feat = g.srcdata['h'][u]  # shape: [num_edges, feature_dim]
        dst_feat = g.dstdata['h'][v]  # shape: [num_edges, feature_dim]
        edge_feat = efeats      # shape: [num_edges, edge_feature_dim]

        # 拼接：[源节点特征，边特征，目标节点特征]
        edge_fin = torch.cat((src_feat, edge_feat, dst_feat), dim=2)

        # （如果你要喂进一个线性层，比如 self.W_edge）
        edge = self.W_edge(edge_fin)
        # u, v = g.edges()
        # edge = self.W_edge(torch.cat((g.srcdata['h'][u], g.dstdata['h'][v]), 2))
        return g.ndata['h'], edge
  
    
class SAGE(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim,  activation):
      super(SAGE, self).__init__()
      # self.atten = nn.Parameter(torch.randn(1, edim))
      self.layers = nn.ModuleList()
      self.layers.append(SAGELayer(ndim_in, edim, ndim_out, F.relu))

    def forward(self, g, nfeats, efeats, corrupt=False):
      # efeats = efeats * self.atten
      if corrupt:
        e_perm = torch.randperm(g.number_of_edges())
        #n_perm = torch.randperm(g.number_of_nodes())
        efeats = efeats[e_perm]
        #nfeats = nfeats[n_perm]
      for i, layer in enumerate(self.layers):
        #nfeats = layer(g, nfeats, efeats)
        nfeats, e_feats = layer(g, nfeats, efeats)
      #return nfeats.sum(1)
      return nfeats.sum(1), e_feats.sum(1)
    
class DGI(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation):
      super(DGI, self).__init__()
      self.encoder = SAGE(ndim_in, ndim_out, edim,  F.relu)
      # self.discriminator = Discriminator(ndim_out)
      self.loss = nn.BCEWithLogitsLoss()

    def forward(self, g, n_features, e_features):
      positive = self.encoder(g, n_features, e_features, corrupt=False)
      negative = self.encoder(g, n_features, e_features, corrupt=True)

      positive = positive[1]
      negative = negative[1]

#       summary = torch.sigmoid(positive.mean(dim=0))

      # positive = self.discriminator(positive, summary)
      # negative = self.discriminator(negative, summary)

      l1 = self.loss(positive,  torch.full_like(positive, 100))
      l2 = self.loss(negative,  torch.full_like(negative, 1))

      return l1 + l2
    
ndim_in = train_g.ndata['h'].shape[2]
# hidden_features = 128
ndim_out = 128
# num_layers = 1
edim = train_g.edata['h'].shape[2]
learning_rate = 1e-3
# epochs = 4000

dgi = DGI(ndim_in,
    ndim_out,
    edim,
    F.relu).to(device)

dgi_optimizer = torch.optim.Adam(dgi.parameters(),
                lr=1e-3,
                weight_decay=0.)

epochs = 51
cnt_wait = 0
best = 1e9
best_t = 0
dur = []
node_features = train_g.ndata['h']
edge_features = train_g.edata['h']

dgi.train()

for epoch in range(epochs):
    
    
    t0 = time.time()

    dgi_optimizer.zero_grad()
    loss = dgi(train_g, node_features, edge_features)
    loss.backward()
    dgi_optimizer.step()

#     if loss < best:
#         best = loss
#         best_t = epoch
#         cnt_wait = 0
#         torch.save(dgi.state_dict(), 'best_dgi.pkl')
#     else:
#         cnt_wait += 1

#     if cnt_wait == patience:
#       print('Early stopping!')
#       break

   
    dur.append(time.time() - t0)

    if epoch % 50 == 0:

        print("Epoch {:05d} | Time(s) {:.4f} | Loss {:.4f} ".format(epoch, np.mean(dur),
              loss.item()))

Epoch 00000 | Time(s) 0.6957 | Loss -0.3424 
Epoch 00050 | Time(s) 0.0984 | Loss -18002.3828 


In [2]:
file_name = "../../DATASET/nf-unsw-v2-0.1.csv"
data = pd.read_csv(file_name)

lab_enc = preprocessing.LabelEncoder()
lab_enc.fit(data["Attack"])

training_emb = dgi.encoder(train_g, train_g.ndata['h'], train_g.edata['h'])[1]
training_emb = training_emb.detach().cpu().numpy()

graphs, _ = dgl.load_graphs('test.bin')
test_g = graphs[0]
test_g = test_g.to(device)

testing_emb = dgi.encoder(test_g, test_g.ndata['h'], test_g.edata['h'])[1]
testing_emb = testing_emb.detach().cpu().numpy()

df_train = pd.DataFrame(training_emb, )
df_train["Attack"] = lab_enc.inverse_transform(
        train_g.edata['Attack'].detach().cpu().numpy())
df_train["Label"] = train_g.edata['Label'].detach().cpu().numpy()

df_test = pd.DataFrame(testing_emb, )
df_test["Attack"] = lab_enc.inverse_transform(
        test_g.edata['Attack'].detach().cpu().numpy())
df_test["Label"] = test_g.edata['Label'].detach().cpu().numpy()

benign_train_samples = df_train[df_train.Label == 0].drop(columns=["Label", "Attack"])
normal_train_samples = df_train.drop(columns=["Label", "Attack"])

train_labels = df_train["Label"]
test_labels = df_test["Label"]

test_samples = df_test.drop(columns=["Label", "Attack"])

contamination = [0.04]

from pyod.models.cblof import CBLOF
n_est = [3]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = CBLOF(n_clusters=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 1/1 [00:05<00:00,  5.03s/it]

{'n_estimators': 3, 'con': 0.04}
0.8160960982793402
              precision    recall  f1-score   support

           0     1.0000    0.9563    0.9777    137714
           1     0.4865    1.0000    0.6545      5704

    accuracy                         0.9580    143418
   macro avg     0.7432    0.9781    0.8161    143418
weighted avg     0.9796    0.9580    0.9648    143418



In [4]:
n_est = [2,3]
# contamination = [0.001, 0.01, 0.04, 0.05, 0.1, 0.2]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = CBLOF(n_clusters=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 2/2 [00:05<00:00,  2.94s/it]

{'n_estimators': 2, 'con': 0.04}
0.8749270170184963
              precision    recall  f1-score   support

           0     0.9907    0.9890    0.9898    137714
           1     0.7450    0.7756    0.7600      5704

    accuracy                         0.9805    143418
   macro avg     0.8679    0.8823    0.8749    143418
weighted avg     0.9809    0.9805    0.9807    143418



In [5]:
from pyod.models.hbos import HBOS

n_est = [5,10]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = HBOS(n_bins=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 2/2 [00:05<00:00,  2.95s/it]

{'n_estimators': 5, 'con': 0.04}
0.9264385476143018
              precision    recall  f1-score   support

           0     1.0000    0.9865    0.9932    137714
           1     0.7539    1.0000    0.8597      5704

    accuracy                         0.9870    143418
   macro avg     0.8769    0.9932    0.9264    143418
weighted avg     0.9902    0.9870    0.9879    143418



In [6]:
n_est = [5,10]
# contamination = [0.001, 0.01, 0.04, 0.05, 0.1, 0.2]
params = list(itertools.product(n_est, contamination))
score = -1
bs = None
for n_est, con in tqdm.tqdm(params):
    
    clf_if = HBOS(n_bins=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 2/2 [00:04<00:00,  2.13s/it]

{'n_estimators': 10, 'con': 0.04}
0.9018957341832948
              precision    recall  f1-score   support

           0     0.9939    0.9897    0.9918    137714
           1     0.7738    0.8541    0.8120      5704

    accuracy                         0.9843    143418
   macro avg     0.8839    0.9219    0.9019    143418
weighted avg     0.9852    0.9843    0.9846    143418



In [8]:
from pyod.models.pca import PCA
n_est = [5,10]
cont = [0.01, 0.04]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = PCA(n_components=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:09<00:00,  2.29s/it]

{'n_estimators': 5, 'con': 0.01}
0.8719697118036361
              precision    recall  f1-score   support

           0     0.9902    0.9891    0.9897    137714
           1     0.7444    0.7644    0.7543      5704

    accuracy                         0.9802    143418
   macro avg     0.8673    0.8768    0.8720    143418
weighted avg     0.9805    0.9802    0.9803    143418



In [10]:
n_est = [5,10]
cont = [0.01, 0.04]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = PCA(n_components=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = y_pred

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 4/4 [00:09<00:00,  2.39s/it]

{'n_estimators': 5, 'con': 0.04}
0.8749270170184963
              precision    recall  f1-score   support

           0     0.9907    0.9890    0.9898    137714
           1     0.7450    0.7756    0.7600      5704

    accuracy                         0.9805    143418
   macro avg     0.8679    0.8823    0.8749    143418
weighted avg     0.9809    0.9805    0.9807    143418



In [11]:
from sklearn.ensemble import IsolationForest
n_est = [20, 50]
cont = [0.01]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = IsolationForest(n_estimators=n_est, contamination=con)
    clf_if.fit(benign_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = list(map(lambda x : 0 if x == 1 else 1, y_pred))

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 2/2 [00:08<00:00,  4.26s/it]

{'n_estimators': 20, 'con': 0.01}
0.8604040543599838
              precision    recall  f1-score   support

           0     0.9876    0.9912    0.9894    137714
           1     0.7676    0.6985    0.7314      5704

    accuracy                         0.9796    143418
   macro avg     0.8776    0.8448    0.8604    143418
weighted avg     0.9788    0.9796    0.9791    143418



In [12]:
from sklearn.ensemble import IsolationForest

n_est = [20, 50]
cont = [ 0.01]
params = list(itertools.product(n_est, cont))
score = -1
bs = None

for n_est, con in tqdm.tqdm(params):
    clf_if = IsolationForest(n_estimators=n_est, contamination=con)
    clf_if.fit(normal_train_samples)
    y_pred = clf_if.predict(test_samples)
    test_pred = list(map(lambda x : 0 if x == 1 else 1, y_pred))

    f1 = f1_score(test_labels, test_pred, average='macro')

    if f1 > score:
        score = f1
        best_params = {'n_estimators': n_est,
                       "con": con
                }
        bs = test_pred
    del clf_if
    gc.collect()


print(best_params)
print(score)
print(classification_report(test_labels, bs, digits=4))

100%|██████████| 2/2 [00:08<00:00,  4.27s/it]

{'n_estimators': 20, 'con': 0.01}
0.6596543365703733
              precision    recall  f1-score   support

           0     0.9685    0.9968    0.9825    137714
           1     0.7376    0.2183    0.3369      5704

    accuracy                         0.9658    143418
   macro avg     0.8530    0.6075    0.6597    143418
weighted avg     0.9594    0.9658    0.9568    143418

